In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"achalks","key":"b31cd80819a5795de2a09b9176ff49f9"}'}

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
100% 80.9M/80.9M [00:00<00:00, 186MB/s]



In [3]:
!unzip -q sentiment140.zip -d sentiment_dataset/
print("Twitter dataset successfully extracted!")

Twitter dataset successfully extracted!


In [4]:
import pandas as pd

# Define the column names as specified on the Kaggle data card
columns = ['target', 'ids', 'date', 'flag', 'user', 'text']

# Load the CSV (using ISO-8859-1 encoding to prevent character errors)
tweet_data = pd.read_csv('sentiment_dataset/training.1600000.processed.noemoticon.csv',
                     encoding='ISO-8859-1',
                     names=columns)

# Display the first 5 rows to ensure it loaded correctly!
tweet_data.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [5]:
import re

# 1. Downsample the data for faster training and to save RAM
print("Sampling down to 100,000 balanced tweets...")
df_negative = tweet_data[tweet_data['target'] == 0].sample(50000, random_state=42)
df_positive = tweet_data[tweet_data['target'] == 4].sample(50000, random_state=42)
tweet_df = pd.concat([df_negative, df_positive])

# 2. Fix the labels (Kaggle labeled positive as '4'. Let's make it '1' for standard binary classification)
tweet_df['target'] = tweet_df['target'].replace(4, 1)

# 3. Create a professional Text Cleaning Pipeline
def clean_text(text):
    text = str(text).lower() # Convert to lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Strip URLs
    text = re.sub(r'\@\w+|\#', '', text) # Strip @mentions and hashtags
    text = re.sub(r'[^\w\s]', '', text) # Strip punctuation
    return text

# 4. Apply the cleaner
print("Scrubbing the text clean (this might take 10-15 seconds)...")
tweet_df['clean_text'] = tweet_df['text'].apply(clean_text)

print("\nData cleaned! Here is the before and after:")
# Display the old messy text next to our new clean text
tweet_df[['target', 'text', 'clean_text']].sample(5)

Sampling down to 100,000 balanced tweets...
Scrubbing the text clean (this might take 10-15 seconds)...

Data cleaned! Here is the before and after:


,target,text,clean_text
242488,0,is so bored of revising criminology when the s...,is so bored of revising criminology when the s...
376594,0,I can't sleep and I don't really want to go to...,i cant sleep and i dont really want to go to m...
92878,0,smh @ my mood be switchin like crazy not in th...,smh my mood be switchin like crazy not in the...
397492,0,who is the real kristen stewart??? please plea...,who is the real kristen stewart please please ...
1367081,1,On joue a &quot;Scene it&quot;,on joue a quotscene itquot


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("1. Splitting data into Training and Testing sets...")
X_train, X_test, y_train, y_test = train_test_split(
    tweet_df['clean_text'],
    tweet_df['target'],
    test_size=0.2, # Save 20% of the data to test the model later
    random_state=42
)

print("2. Translating words into numbers (Vectorization)...")
# We limit to the top 10,000 most common words to keep it fast and efficient
vectorizer = TfidfVectorizer(max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("3. Training the Logistic Regression model...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

print("4. Testing the model on unseen data...")
predictions = model.predict(X_test_vec)

# Display the final performance metrics
accuracy = accuracy_score(y_test, predictions)
print(f"\n✅ Final Model Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Classification Report:")
print(classification_report(y_test, predictions))

1. Splitting data into Training and Testing sets...
2. Translating words into numbers (Vectorization)...
3. Training the Logistic Regression model...
4. Testing the model on unseen data...

✅ Final Model Accuracy: 78.30%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.77      0.78     10035
           1       0.78      0.79      0.78      9965

    accuracy                           0.78     20000
   macro avg       0.78      0.78      0.78     20000
weighted avg       0.78      0.78      0.78     20000



In [12]:
# Create a function to test our own custom tweets
def predict_sentiment(custom_text):
    # 1. Clean the text using the exact same function from earlier
    cleaned = clean_text(custom_text)

    # 2. Convert the words to numbers using our fitted vectorizer
    vectorized = vectorizer.transform([cleaned])

    # 3. Make the prediction
    prediction = model.predict(vectorized)[0]

    # 4. Print the result!
    if prediction == 1:
        print(f"Text: '{custom_text}'\nPrediction: 🟢 POSITIVE\n")
    else:
        print(f"Text: '{custom_text}'\nPrediction: 🔴 NEGATIVE\n")

# Let's test it with a few examples!
print("Testing the AI's intuition...\n" + "-"*30)

predict_sentiment("I absolutely loved the new Batman movie, the cinematography was amazing!")
predict_sentiment("My flight was delayed by 4 hours and they lost my luggage. I am furious.")
predict_sentiment("I had a great time!!")

# Try adding your own string below to see what it predicts!
# predict_sentiment("Type your own sentence right here!")

Testing the AI's intuition...
------------------------------
Text: 'I absolutely loved the new Batman movie, the cinematography was amazing!'
Prediction: 🟢 POSITIVE

Text: 'My flight was delayed by 4 hours and they lost my luggage. I am furious.'
Prediction: 🔴 NEGATIVE

Text: 'I had a great time!!'
Prediction: 🟢 POSITIVE



In [13]:
import pickle

# Save the trained model
with open('sentiment_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save the vectorizer
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Model and Vectorizer successfully saved to Colab disk!")

Model and Vectorizer successfully saved to Colab disk!
